# Qwen2-VL prompting on PathEval (Kaggle runner)

Thin runner for `src/prompting.py`: clones the team repo, loads Qwen2-VL-7B once, runs prompting
conditions, and summarizes. All logic lives in the repo; this notebook only calls it.

**Settings:** Accelerator = **GPU T4 x2**, Internet = On.
**Inputs:** add the `patheval-2d-test` dataset (Add Input → Your Datasets).
**Secrets (Add-ons → Secrets):** `GITHUB_TOKEN`, a classic GitHub token with `repo` scope (the team repo is private).

## 1. Settings

In [ ]:
REPO = "zoekrishtul/Deep-Learning-Final-Project"
BRANCH = "gaurav/prompting"        # change to "main" once merged
STRATEGIES = ["direct", "step_by_step", "structured"]   # which prompting strategies to run
RUN_SWAPPED = True                 # also run each strategy with left/right halves swapped
N_PER_SCENARIO = 10                # 10 -> 150 samples; None -> all 1050
BATCH_SIZE = 8                     # direct ~2 s/sample; reasoning prompts are several times slower
OUT_DIR = "/kaggle/working/results"

## 2. Get the code
Cloned to `/tmp` (not `/kaggle/working`) so the token never ends up in saved notebook output.

In [ ]:
import os, sys, subprocess
os.chdir("/kaggle/working")        # step out of /tmp/repo before deleting it, so rerunning is safe
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None
    print("No GITHUB_TOKEN secret; cloning will only work if the repo is public")

def git(*args):
    """Run git; on failure, raise an error with the token masked (never print the URL)."""
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        msg = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git {args[0]} failed:\n{msg}")
    return result.stdout

auth = f"x-access-token:{token}@" if token else ""
subprocess.run(["rm", "-rf", "/tmp/repo"], check=True)
git("clone", "-q", "--branch", BRANCH, f"https://{auth}github.com/{REPO}.git", "/tmp/repo")
git("-C", "/tmp/repo", "remote", "set-url", "origin", f"https://github.com/{REPO}.git")  # drop token
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")
print(git("log", "--oneline", "-1"))

## 3. Find the data
Uses `test_2D.jsonl` if the dataset has it; otherwise converts the older `test_2D_clean.json`.

In [ ]:
import glob, json
from src.data import load_manifest, manifest_from_clean, write_jsonl

found = (glob.glob("/kaggle/input/**/test_2D.jsonl", recursive=True) or
         glob.glob("/kaggle/input/**/test_2D_clean.json", recursive=True))
if not found:
    raise FileNotFoundError("No PathEval data under /kaggle/input. Attach the dataset: "
                            "right panel -> Add Input -> Your Datasets -> patheval-2d-test")
DATA_ROOT = os.path.dirname(found[0])          # image paths are relative to this folder
if found[0].endswith(".jsonl"):
    MANIFEST = found[0]
else:                                          # older dataset version: convert once
    MANIFEST = "/kaggle/working/test_2D.jsonl"
    with open(found[0]) as f:
        write_jsonl(manifest_from_clean(json.load(f)), MANIFEST)
rows = load_manifest(MANIFEST)
print(len(rows), "samples | manifest:", MANIFEST, "| data root:", DATA_ROOT)

## 4. Load Qwen2-VL-7B (float16, split across both T4s)

In [ ]:
from src.prompting import QwenPathEvaluator, run, summarize, print_summary
evaluator = QwenPathEvaluator()
evaluator.describe()

## 5. Run the prompting conditions
Each run resumes if its output file already exists, so rerunning this cell after a disconnect
continues where it stopped. The progress line shows seconds per sample and time left.

In [ ]:
outputs = {}
for strategy in STRATEGIES:
    for swap in ([False, True] if RUN_SWAPPED else [False]):
        name = f"{strategy}_2D" + ("_swapped" if swap else "")
        outputs[(strategy, swap)] = run(MANIFEST, DATA_ROOT, f"{OUT_DIR}/{name}.jsonl", evaluator,
                                        strategy=strategy, swap=swap,
                                        n_per_scenario=N_PER_SCENARIO, batch_size=BATCH_SIZE)

## 6. Quick summary (official metrics: `src/evaluation.py`)

In [ ]:
for strategy in STRATEGIES:
    print(f"==================== {strategy} ====================")
    print_summary(summarize(outputs[(strategy, False)],
                            swapped_path=outputs.get((strategy, True))))
    print()

## 7. Look at a few reasoning outputs

In [ ]:
import random
from src.data import read_jsonl
for strategy in STRATEGIES[1:]:
    rows = read_jsonl(outputs[(strategy, False)])
    for r in random.Random(0).sample(rows, 2):
        print(f"--- {strategy} | sample {r['sample_id']} | label {r['label']} | parsed {r['prediction']}")
        print(r["raw_response"][-1200:], "\n")

Download `/kaggle/working/results/` from the Output panel before stopping the session
(results are not committed to Git; see `results/README.md`).